# 04 — Player Performance & Season Outcome Models

Four sub-models predicting individual and season-level outcomes:

1. **Player Performance** — Next-season PTS, AST, REB per game
2. **Win Totals** — Regular season wins per team (XGBoost + Ridge blend)
3. **Championship Odds** — Derived from predicted wins + SRS
4. **MVP Race** — Award share prediction (Spearman \u03c1 = 0.89)

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.stats import spearmanr

from nba_predict.config import MODELS_DIR, TRAIN_SEASONS, VAL_SEASONS, TEST_SEASONS
from nba_predict.models.player_performance import _build_player_dataset, get_feature_columns as get_player_features
from nba_predict.models.season_outcomes import (
    _build_win_totals_dataset, _build_mvp_dataset,
    _get_win_totals_features, _get_mvp_features,
    _compute_championship_odds,
)
from nba_predict.evaluation.metrics import regression_metrics, feature_importance

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries loaded.')

Libraries loaded.


## 1. Player Performance

In [2]:
# Load model and data
pp_artifact = joblib.load(MODELS_DIR / 'player_performance.joblib')
pp_models = pp_artifact['models']
pp_features = pp_artifact['feature_cols']

player_df = _build_player_dataset()
test_players = player_df[player_df['season'].isin(TEST_SEASONS)]

# Use only features that exist in both the model and the data
avail_features = [f for f in pp_features if f in test_players.columns]
X_test = test_players[avail_features].astype(float)

print(f"Player-seasons in test: {len(test_players)}")
print(f"Features: {len(avail_features)}")

Player-seasons in test: 729
Features: 31


In [3]:
# Predicted vs Actual for all 3 targets
targets = {'pts': 'target_pts', 'ast': 'target_ast', 'reb': 'target_reb'}
prior_map = {'pts': 'pts_per_g', 'ast': 'ast_per_g', 'reb': 'trb_per_g'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, target_col) in enumerate(targets.items()):
    y_true = pd.to_numeric(test_players[target_col], errors='coerce').values
    y_pred = pp_models[name].predict(X_test)
    y_prior = pd.to_numeric(test_players[prior_map[name]], errors='coerce').values

    mask = ~np.isnan(y_true) & ~np.isnan(y_prior)
    model_mae = np.abs(y_true[mask] - y_pred[mask]).mean()
    baseline_mae = np.abs(y_true[mask] - y_prior[mask]).mean()
    improvement = (baseline_mae - model_mae) / baseline_mae * 100

    axes[i].scatter(y_true, y_pred, alpha=0.3, s=10, color='steelblue')
    lims = [0, max(np.nanmax(y_true), np.nanmax(y_pred)) * 1.1]
    axes[i].plot(lims, lims, 'r--', alpha=0.5)
    axes[i].set_xlabel('Actual')
    axes[i].set_ylabel('Predicted')
    axes[i].set_title(f'{name.upper()} (MAE={model_mae:.2f}, {improvement:+.1f}% vs baseline)')

plt.suptitle('Player Performance: Predicted vs Actual', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41313/3864372763.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Feature importance comparison across targets
fi_data = {}
for name, model in pp_models.items():
    fi = feature_importance(model, avail_features, top_n=10)
    fi_data[name] = fi.set_index('feature')['importance']

fi_df = pd.DataFrame(fi_data).fillna(0)

fig, ax = plt.subplots(figsize=(12, 8))
fi_df.plot(kind='barh', ax=ax, width=0.8)
ax.set_xlabel('Importance')
ax.set_title('Feature Importance by Target (Top 10 per model)')
ax.legend(title='Target')
plt.tight_layout()
plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41313/1822348435.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Error by age
test_analysis = test_players.copy()
test_analysis['age'] = pd.to_numeric(test_analysis['age'], errors='coerce')
test_analysis['pts_error'] = np.abs(
    pd.to_numeric(test_analysis['target_pts'], errors='coerce') - pp_models['pts'].predict(X_test)
)
test_analysis['age_bucket'] = pd.cut(
    test_analysis['age'], bins=[18, 23, 27, 31, 40],
    labels=['19-23', '24-27', '28-31', '32+']
)

fig, ax = plt.subplots(figsize=(8, 5))
test_analysis.groupby('age_bucket', observed=True)['pts_error'].mean().plot(
    kind='bar', ax=ax, color='steelblue', alpha=0.8
)
ax.set_title('PTS Prediction Error by Age Group')
ax.set_ylabel('MAE')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41313/935808174.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Win Totals

In [6]:
so_artifact = joblib.load(MODELS_DIR / 'season_outcomes.joblib')
wt_model = so_artifact['win_totals_model']
ridge = so_artifact.get('win_totals_ridge')
scaler = so_artifact.get('win_totals_scaler')
wt_features = so_artifact['win_totals_features']

wt_df = _build_win_totals_dataset()
test_wt = wt_df[wt_df['season'].isin(TEST_SEASONS)]
avail_wt_features = [c for c in wt_features if c in test_wt.columns]
X_wt = test_wt[avail_wt_features].astype(float)
y_wt = pd.to_numeric(test_wt['target_wins'], errors='coerce').astype(float)

# Blended prediction
xgb_pred = wt_model.predict(X_wt)
if ridge and scaler:
    ridge_pred = ridge.predict(scaler.transform(X_wt.fillna(0)))
    wt_pred = 0.4 * xgb_pred + 0.6 * ridge_pred
else:
    wt_pred = xgb_pred

wt_metrics = regression_metrics(y_wt.values, wt_pred)
baseline_mae = np.abs(y_wt.values - pd.to_numeric(test_wt['prev_wins'], errors='coerce').values).mean()

print(f"Win Totals MAE:  {wt_metrics['mae']:.2f} wins")
print(f"Baseline MAE:    {baseline_mae:.2f} wins")

Win Totals MAE:  9.60 wins
Baseline MAE:    8.80 wins


In [7]:
# Predicted vs Actual wins with team labels
fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(y_wt.values, wt_pred, s=50, color='steelblue', alpha=0.7, zorder=5)
for idx_pos, (_, row) in enumerate(test_wt.iterrows()):
    if idx_pos < len(wt_pred):
        ax.annotate(row['team'], (y_wt.values[idx_pos], wt_pred[idx_pos]),
                    fontsize=7, alpha=0.7)

lims = [10, 70]
ax.plot(lims, lims, 'r--', alpha=0.5)
ax.set_xlabel('Actual Wins')
ax.set_ylabel('Predicted Wins')
ax.set_title(f'Win Totals: Predicted vs Actual (MAE={wt_metrics["mae"]:.1f})')
ax.set_xlim(lims)
ax.set_ylim(lims)
plt.tight_layout()
plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41313/3509800540.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Championship Odds

In [8]:
test_wt_copy = test_wt.copy()
test_wt_copy['pred_wins'] = wt_pred
champ_df = _compute_championship_odds(test_wt_copy)

for season in sorted(test_wt_copy['season'].unique()):
    season_top = champ_df[champ_df['season'] == season].head(10)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(season_top['team'], season_top['championship_prob'],
            color='steelblue', alpha=0.8)
    ax.set_xlabel('Championship Probability')
    ax.set_title(f'{int(season)} \u2014 Top 10 Championship Contenders')
    ax.invert_yaxis()
    for i, (_, row) in enumerate(season_top.iterrows()):
        ax.text(row['championship_prob'] + 0.005, i,
                f"{row['championship_prob']:.1%}", va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41313/629721823.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41313/629721823.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. MVP Race

In [9]:
mvp_model = so_artifact['mvp_model']
mvp_features = so_artifact['mvp_features']

mvp_df = _build_mvp_dataset()
test_mvp = mvp_df[mvp_df['season'].isin(TEST_SEASONS)]
avail_mvp_features = [c for c in mvp_features if c in test_mvp.columns]
X_mvp = test_mvp[avail_mvp_features].astype(float)
y_mvp = pd.to_numeric(test_mvp['target_award_share'], errors='coerce').astype(float)

mvp_pred = mvp_model.predict(X_mvp)
mvp_metrics = regression_metrics(y_mvp.values, mvp_pred)

voted = y_mvp.values > 0
rho, p_val = spearmanr(y_mvp.values[voted], mvp_pred[voted])

print(f"MVP Award Share MAE:  {mvp_metrics['mae']:.4f}")
print(f"Spearman \u03c1 (voted):   {rho:.4f} (p={p_val:.2e})")

MVP Award Share MAE:  0.0107
Spearman ρ (voted):   0.8935 (p=4.97e-08)


In [10]:
# Top MVP candidates per test season
test_mvp_copy = test_mvp.copy()
test_mvp_copy['pred_share'] = mvp_pred

for season in sorted(test_mvp_copy['season'].unique()):
    top = test_mvp_copy[test_mvp_copy['season'] == season].nlargest(10, 'pred_share')
    
    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(top))
    width = 0.35
    ax.bar([i - width/2 for i in x], top['pred_share'], width,
           label='Predicted', color='steelblue', alpha=0.8)
    ax.bar([i + width/2 for i in x], pd.to_numeric(top['target_award_share'], errors='coerce'), width,
           label='Actual', color='#F44336', alpha=0.8)
    ax.set_xticks(list(x))
    # Use player name column
    name_col = 'player' if 'player' in top.columns else 'name_display'
    ax.set_xticklabels(top[name_col].values, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Award Share')
    ax.set_title(f'{int(season)} MVP Race \u2014 Predicted vs Actual')
    ax.legend()
    plt.tight_layout()
    plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41313/1839600596.py:22: UserWarning: Glyph 135 (\x87) missing from font(s) Arial.
  plt.tight_layout()
/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41313/1839600596.py:22: UserWarning: Glyph 141 (\x8d) missing from font(s) Arial.
  plt.tight_layout()
/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41313/1839600596.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41313/1839600596.py:22: UserWarning: Glyph 135 (\x87) missing from font(s) Arial.
  plt.tight_layout()
/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41313/1839600596.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# MVP feature importance
fi_mvp = feature_importance(mvp_model, avail_mvp_features, top_n=15)

fig, ax = plt.subplots(figsize=(10, 6))
fi_plot = fi_mvp.iloc[::-1]
ax.barh(fi_plot['feature'], fi_plot['importance'], color='teal', alpha=0.8)
ax.set_xlabel('Importance (gain)')
ax.set_title('MVP Model \u2014 Top 15 Features')
plt.tight_layout()
plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41313/921205201.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Summary

| Model | Metric | Value | vs Baseline |
|-------|--------|-------|-------------|
| Player PTS | MAE | 2.29 | +7.7% better |
| Player AST | MAE | 0.66 | +9.1% better |
| Player REB | MAE | 0.80 | +0.8% better |
| Win Totals | MAE | 9.60 | -9.1% (hard problem, N=600) |
| MVP Race | Spearman \u03c1 | 0.894 | Excellent rank correlation |

**Key insights:**
- VORP is the #1 MVP feature
- Older players (32+) have higher prediction error due to retirement risk
- Win totals remains the hardest problem \u2014 only ~600 training samples